# 2.3 无监督学习 (Unsupervised Learning)

在没有标签或预定义任务的情况下发现数据的内在结构和模式。

本节涵盖：
- 语言建模
- 聚类
- 主题建模
- 数据分布估计

## 1. 语言建模

**目的**：学习自然语言的概率分布

**基本原理**：通过建模P(x)来学习语言数据的分布。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SimpleLM(nn.Module):
    def __init__(self, vocab_size=500, d_model=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.rnn = nn.LSTM(d_model, d_model, batch_first=True)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        h, _ = self.rnn(self.embed(x))
        return self.head(h)
    
    def compute_nll(self, x):
        logits = self(x[:, :-1])
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1))
        return loss

model = SimpleLM()
data = torch.randint(0, 500, (16, 32))
nll = model.compute_nll(data)
ppl = torch.exp(nll)

print('=== Language Modeling ===')
print(f'NLL Loss: {nll.item():.4f}')
print(f'Perplexity: {ppl.item():.2f}')
print(f'\nPerplexity = exp(NLL), measures how well the model predicts the next token.')
print(f'Lower PPL = better model. Random baseline PPL = vocab_size = {500}')

## 2. 聚类

**目的**：发现数据中的自然分组

**基本原理**：使用K-Means等方法将相似的数据点分组。

In [ ]:
import torch
from collections import Counter

def kmeans(X, k=3, max_iters=100):
    """Simple K-Means clustering."""
    n = X.size(0)
    centroids = X[torch.randperm(n)[:k]].clone()
    
    for i in range(max_iters):
        dists = torch.cdist(X, centroids)
        assignments = dists.argmin(dim=1)
        
        new_centroids = torch.zeros_like(centroids)
        for j in range(k):
            mask = assignments == j
            if mask.any():
                new_centroids[j] = X[mask].mean(dim=0)
            else:
                new_centroids[j] = centroids[j]
        
        if torch.allclose(centroids, new_centroids, atol=1e-6):
            break
        centroids = new_centroids
    
    return assignments, centroids

torch.manual_seed(42)
n_per_cluster = 100
centers = torch.tensor([[1.0, 1.0], [5.0, 5.0], [9.0, 1.0]])
X = torch.cat([c + torch.randn(n_per_cluster, 2) * 0.5 for c in centers])

assignments, centroids = kmeans(X, k=3)

print('=== K-Means Clustering ===')
print(f'Data: {X.shape[0]} points in {X.shape[1]}D')
print(f'Cluster sizes: {dict(Counter(assignments.tolist()))}')
for i, c in enumerate(centroids):
    print(f'Centroid {i}: ({c[0]:.2f}, {c[1]:.2f})')
print(f'\nApplication: Cluster text embeddings to discover topic groups in unlabeled data.')

## 3. 主题建模

**目的**：发现文档集合中的潜在主题

**基本原理**：使用LDA、BERTopic等方法从文档集合中提取潜在主题分布。

In [ ]:
import random
from collections import Counter, defaultdict

random.seed(42)

topics = {
    'tech': ['algorithm', 'software', 'computer', 'data', 'network', 'code', 'system', 'digital'],
    'health': ['patient', 'disease', 'treatment', 'doctor', 'medicine', 'hospital', 'symptom', 'diagnosis'],
    'finance': ['market', 'stock', 'investment', 'bank', 'profit', 'economy', 'trade', 'revenue'],
}

n_docs = 100
docs = []
doc_topics = []
for _ in range(n_docs):
    mix = random.choice(['pure', 'mixed'])
    if mix == 'pure':
        t = random.choice(list(topics.keys()))
    else:
        t = random.choice(list(topics.keys()))
    words = random.choices(topics[t], k=20)
    docs.append(words)
    doc_topics.append(t)

word_topic_counts = defaultdict(Counter)
for doc, topic in zip(docs, doc_topics):
    for word in set(doc):
        word_topic_counts[word][topic] += 1

print('=== Simple Topic Modeling ===')
print(f'Generated {n_docs} documents across {len(topics)} topics')
print(f'\nTop words per topic (by document frequency):')
for topic in topics:
    topic_words = [(w, counts[topic]) for w, counts in word_topic_counts.items() if topic in counts]
    topic_words.sort(key=lambda x: -x[1])
    print(f'  {topic:10s}: {[(w, c) for w, c in topic_words[:5]]}')

## 4. 数据分布估计

**目的**：估计数据的统计特性

**基本原理**：使用核密度估计、流模型等方法估计数据的概率密度。

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

class SimpleFlow(nn.Module):
    """Simple affine coupling layer for normalizing flow."""
    def __init__(self, dim=2, hidden_dim=32):
        super().__init__()
        self.dim = dim
        self.half = dim // 2
        self.net = nn.Sequential(
            nn.Linear(self.half, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, dim)
        )
    
    def forward(self, x):
        x1, x2 = x[:, :self.half], x[:, self.half:]
        params = self.net(x1)
        s, t = params[:, :self.half], params[:, self.half:]
        s = torch.tanh(s)
        y2 = x2 * torch.exp(s) + t
        log_det = s.sum(dim=-1)
        return torch.cat([x1, y2], dim=-1), log_det
    
    def inverse(self, y):
        y1, y2 = y[:, :self.half], y[:, self.half:]
        params = self.net(y1)
        s, t = params[:, :self.half], params[:, self.half:]
        s = torch.tanh(s)
        x2 = (y2 - t) * torch.exp(-s)
        return torch.cat([y1, x2], dim=-1)

flow = SimpleFlow(dim=2)
target_data = torch.randn(500, 2) * 0.5 + torch.tensor([2.0, 3.0])
base_samples = torch.randn(100, 2)

transformed, log_det = flow(base_samples)
reconstructed = flow.inverse(transformed)

print('=== Normalizing Flow for Density Estimation ===')
print(f'Base samples (standard normal): mean={base_samples.mean(0).tolist()}, std={base_data.std(0).tolist() if False else "~1.0"}')
print(f'Transformed samples: mean={transformed.mean(0).detach().tolist()}, std={transformed.std(0).detach().tolist()}')
print(f'Reconstruction error: {(reconstructed - base_samples).abs().mean().item():.6f}')
print(f'\nFlow models learn invertible transformations to estimate exact log-likelihood.')